# SHIPP Task 93 — Deployed End-to-End Validation

**Purpose:** close the remaining P0 proof after Task 92 by validating one real workflow executed through the deployed Databricks App.

This notebook is **read-only for business state**. It does not create Listings, Requests, Saves, or rejection fixtures. Those actions must happen through the deployed App / controlled final environment first.

Required App evidence before running:
1. App `Deployment & permissions evidence` shows PASS.
2. Donor publishes a Listing with photo + description; copy `listing_id`.
3. Requester creates a Request; copy `request_id`.
4. Run Tasks 90 and 91.
5. In deployed App, exercise Agent READ and save the selected Listing; copy `saved_item_id`.
6. Repeat save to produce `REJECTED_DUPLICATE`.
7. Use a controlled stale/unavailable candidate and attempt save to produce `REJECTED_UNAVAILABLE`; copy that `listing_id`.
8. Run Velocity validation for the Listing or Request.
9. Ensure one persisted ORS retry/failure evidence row exists.


In [0]:
%run ../common/shipp_silver_lib

In [0]:
import json
import uuid
from datetime import datetime, timezone

from pyspark.sql import Row

from config.tables import TABLES
from data_pipeline.quality.deployed_e2e_validation import run_deployed_e2e_validation

dbutils.widgets.text('listing_id', '', 'Listing ID created in deployed App')
dbutils.widgets.text('request_id', '', 'Request ID created in deployed App')
dbutils.widgets.text('saved_item_id', '', 'Saved Item ID returned by deployed App')
dbutils.widgets.text('unavailable_listing_id', '', 'Listing ID rejected as unavailable')

LISTING_ID = dbutils.widgets.get('listing_id').strip()
REQUEST_ID = dbutils.widgets.get('request_id').strip()
SAVED_ITEM_ID = dbutils.widgets.get('saved_item_id').strip()
UNAVAILABLE_LISTING_ID = dbutils.widgets.get('unavailable_listing_id').strip()

required_ids = {
    'listing_id': LISTING_ID,
    'request_id': REQUEST_ID,
    'saved_item_id': SAVED_ITEM_ID,
    'unavailable_listing_id': UNAVAILABLE_LISTING_ID,
}
missing = [name for name, value in required_ids.items() if not value]
assert not missing, f'BLOCKED — fill App evidence widgets: {missing}'

tables = dict(TABLES)
tables['velocity_measurements'] = 'bootcamp_students.shipp_gold.velocity_measurements'

summary, checks = run_deployed_e2e_validation(
    spark,
    tables,
    listing_id=LISTING_ID,
    request_id=REQUEST_ID,
    saved_item_id=SAVED_ITEM_ID,
    unavailable_listing_id=UNAVAILABLE_LISTING_ID,
)

print(json.dumps(summary, indent=2, default=str))
display(spark.createDataFrame([Row(**check.__dict__) for check in checks]))


In [0]:
# Persist one immutable evidence row per check.
RUN_ID = str(uuid.uuid4())
CHECKED_AT = datetime.now(timezone.utc).replace(tzinfo=None)
LOG_TABLE = 'bootcamp_students.shipp_gold.deployed_e2e_validation_log'

rows = [
    (
        RUN_ID,
        LISTING_ID,
        REQUEST_ID,
        SAVED_ITEM_ID,
        UNAVAILABLE_LISTING_ID,
        check.check_name,
        check.status,
        check.observed_value,
        check.detail,
        summary['status'],
        CHECKED_AT,
    )
    for check in checks
]

schema = '''
validation_run_id string,
listing_id string,
request_id string,
saved_item_id string,
unavailable_listing_id string,
check_name string,
check_status string,
observed_value string,
detail string,
overall_status string,
checked_at timestamp
'''

spark.sql('CREATE SCHEMA IF NOT EXISTS bootcamp_students.shipp_gold')
spark.createDataFrame(rows, schema).write.format('delta').mode('append').saveAsTable(LOG_TABLE)

print('validation_run_id =', RUN_ID)
print('log_table =', LOG_TABLE)
print('overall_status =', summary['status'])
print('failed_checks =', summary['failed_checks'])

if summary['status'] != 'PASS':
    raise AssertionError(
        'FAILED — deployed E2E validation has unresolved checks: '
        + ', '.join(summary['failed_checks'])
    )

print('PASS — deployed SHIPP vertical slice validated end to end.')
dbutils.notebook.exit(json.dumps({
    'validation_run_id': RUN_ID,
    'status': summary['status'],
    'passed_checks': summary['passed_checks'],
    'total_checks': summary['total_checks'],
    'failed_checks': summary['failed_checks'],
}, default=str))
